# Coffee Standard J25 — YOLOv8n CWCF Target-Attribute Audit

Validation-only diagnosis of V8N_MATCHED vs V8N_CWCF1 for Biji Hitam Pecah. No training. Locked test remains closed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys
from pathlib import Path
BRANCH='codex/j25-yolov8n-cwcf-target-audit'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
experiment_roots=[Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments'),*Path('/content/drive/.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection/experiments')]
def find_file(folder, relative):
    matches=[root/folder/relative for root in experiment_roots if (root/folder/relative).is_file()]
    if not matches: raise FileNotFoundError(f'{folder}/{relative} tidak ditemukan di Drive')
    return matches[0]
MATCHED_RESULT=find_file('coffee-standard-j25-yolov8n-matched-v1',Path('val_reports/V8N_MATCHED_seed42_result.json'))
MATCHED_CKPT=find_file('coffee-standard-j25-yolov8n-matched-v1',Path('V8N_MATCHED/V8N_MATCHED_seed42/weights/best.pt'))
CWCF1_RESULT=find_file('coffee-standard-j25-yolov8n-cwcf-v1',Path('val_reports/V8N_CWCF1_seed42_result.json'))
CWCF1_CKPT=find_file('coffee-standard-j25-yolov8n-cwcf-v1',Path('V8N_CWCF1/V8N_CWCF1_seed42/weights/best.pt'))
PROJECT=CWCF1_RESULT.parents[3]
print('MATCHED:',MATCHED_CKPT)
print('CWCF1:',CWCF1_CKPT)
print('PROJECT:',PROJECT)


In [ ]:
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
assert contract['decision']=='PASS'
assert contract['test_images_extracted'] is False
print('DATA:',contract['images'],'| INSTANCES:',contract['instances'],'| TEST EXTRACTED:',contract['test_images_extracted'])


In [ ]:
from coffee_detector.analysis.coffee_standard_j25_yolov8n_cwcf_target_audit import run_audit
OUT=PROJECT/'experiments/coffee-standard-j25-yolov8n-cwcf-target-audit-v1'
OUT.mkdir(parents=True,exist_ok=True)
OUTPUT=OUT/'V8N_MATCHED_vs_CWCF1_target_attribute_audit.json'
audit=run_audit(DATA,CONTRACT,PROVENANCE,MATCHED_CKPT,MATCHED_RESULT,CWCF1_CKPT,CWCF1_RESULT,OUTPUT,device='0',authorize_diagnostic=True)
print('SAVED:',OUTPUT)
print('TEST OPENED:',audit['test_opened'])


In [ ]:
support=audit['support_context']
matched=audit['models']['V8N_MATCHED']['target_summary']
cwcf=audit['models']['V8N_CWCF1']['target_summary']
print('SUPPORT:',json.dumps(support,indent=2,ensure_ascii=False))
print('MATCHED TARGET:',json.dumps(matched,indent=2,ensure_ascii=False))
print('CWCF1 TARGET:',json.dumps(cwcf,indent=2,ensure_ascii=False))
print('CWCF1 FAMILY:',json.dumps(audit['models']['V8N_CWCF1']['confusion_family_summary'],indent=2,ensure_ascii=False))
print('PAIRED TARGET:')
for row in audit['paired_target_comparison']: print(json.dumps(row,ensure_ascii=False))
